# Étape 1 — Exploration et Préparation des Données
## Projet : Prédiction du Prix des Logements (House Prices — Ames, Iowa)

Ce notebook couvre **intégralement** l'Étape 1 du projet :
1. Analyse de la structure et des types de données
2. Identification des valeurs manquantes et des doublons
3. Recherche des incohérences et valeurs aberrantes
4. Distinction variables numériques vs catégorielles
5. Analyse approfondie de `SalePrice` (variable cible)
6. Traitement complet des valeurs manquantes
7. Feature Engineering (nouvelles colonnes synthétiques)
8. Encodage des variables catégorielles
9. Normalisation et standardisation
10. Séparation X / y — sans fuite de données (Data Leakage)

> **Important :** Le prétraitement est conçu pour éviter toute fuite de données.
> Le `train_test_split` est effectué **avant** tout calcul de statistiques (moyennes, médianes)
> afin que les données de test n'influencent jamais l'entraînement du modèle.


## 0. Importation des Bibliothèques

Nous chargeons toutes les bibliothèques nécessaires pour ce notebook.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, OrdinalEncoder, OneHotEncoder
)
from sklearn.impute import SimpleImputer
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("Toutes les bibliothèques sont chargées avec succès !")


Toutes les bibliothèques sont chargées avec succès !


## 1. Analyse de la Structure et des Types de Données

### 1.1 Chargement du fichier

On commence par charger le fichier CSV dans un DataFrame pandas.
Ensuite, on examine la taille de la table, les noms des colonnes et leurs types.


In [ ]:
# Chargement des données
df = pd.read_csv('House_Prices.csv')

# La colonne Id est juste un numéro de ligne, elle ne contient aucune information utile
# On la supprime immédiatement pour éviter qu'elle perturbe les modèles
df.drop(columns=['Id'], inplace=True)

print(f"Dimensions : {df.shape[0]} lignes (maisons) x {df.shape[1]} colonnes (caractéristiques)")
print(f"\nNom des colonnes :\n{df.columns.tolist()}")


In [ ]:
# Aperçu des 5 premières lignes de la table
df.head()


In [ ]:
# Types de données par colonne et nombre de valeurs non nulles
print("=== Informations générales sur le dataset ===\n")
df.info()


In [ ]:
# Répartition par type de donnée
types = df.dtypes.value_counts().reset_index()
types.columns = ['Type de Données', 'Nombre de Colonnes']
print(types.to_string(index=False))


## 2. Identification des Doublons

Un doublon est une ligne qui serait une copie exacte d'une autre.
Dans un jeu de données immobilier (chaque vente est unique), les doublons
seraient le signe d'une erreur de saisie ou d'importation.


In [ ]:
nb_doublons = df.duplicated().sum()
print(f"Nombre de lignes en double : {nb_doublons}")

if nb_doublons == 0:
    print("=> Aucun doublon détecté. Le jeu de données est propre sur ce point.")
else:
    print(f"=> ATTENTION : {nb_doublons} doublons détectés. On les supprime.")
    df = df.drop_duplicates()
    print(f"=> Nouvelles dimensions après suppression : {df.shape}")


## 3. Identification des Valeurs Manquantes

Une valeur manquante (affichée NaN dans pandas) peut signifier deux choses
très différentes dans ce jeu de données :

- **Absence d'un équipement** : Pour la colonne `PoolQC` (Qualité de la piscine),
  une case vide signifie simplement que la maison n'a pas de piscine.
  Ce n'est PAS une donnée perdue, c'est une information utile : "pas de piscine".
  On la remplacera par la chaîne de caractères `'None'` (ou `0` si numérique).

- **Oubli de saisie** : Pour `LotFrontage` (façade sur rue), certaines maisons
  n'ont pas eu leur façade mesurée. C'est un vrai oubli qu'on imputera intelligemment.

Commençons par identifier toutes les colonnes concernées.


In [1]:
# Tableau complet des valeurs manquantes
nulls = df.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
pct = (nulls / len(df) * 100).round(2)

df_nulls = pd.DataFrame({
    'Nb Valeurs Manquantes': nulls,
    'Pourcentage (%)': pct
})

print(f"Colonnes avec au moins une valeur manquante : {len(df_nulls)} sur {df.shape[1]}")
print("\n", df_nulls.to_string())


NameError: name 'df' is not defined

In [ ]:
# Graphique des valeurs manquantes
plt.figure(figsize=(13, 7))
colors = ['#d62728' if p > 20 else '#ff7f0e' if p > 5 else '#1f77b4'
          for p in df_nulls['Pourcentage (%)']]

bars = plt.barh(df_nulls.index[::-1], df_nulls['Pourcentage (%)'][::-1],
                color=colors[::-1], edgecolor='white')

plt.xlabel('Pourcentage de Valeurs Manquantes (%)')
plt.title('Valeurs Manquantes par Colonne\n'
          '(Rouge > 20% | Orange > 5% | Bleu < 5%)',
          fontsize=13, fontweight='bold')

for bar in bars:
    w = bar.get_width()
    if w > 0.5:
        plt.text(w + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{w:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()


## 4. Recherche des Incohérences et Valeurs Aberrantes (Outliers)

Une valeur aberrante est une observation qui s'éloigne beaucoup de la majorité.
Elle peut indiquer :
- Une erreur de saisie (ex: une maison avec 50 salles de bain).
- Un cas exceptionnel réel (ex: une villa de luxe).

Pour ce jeu de données, deux anomalies célèbres sont documentées :
- Deux maisons ont une très grande surface habitable (> 4 000 pieds carrés)
  mais ont été vendues à des prix anormalement bas (< 200 000 $).
  Ce sont des ventes particulières (familiales ou judiciaires) qui fausseraient
  l'entraînement du modèle.


In [ ]:
# Statistiques descriptives des colonnes numériques
df.describe().T.sort_values('std', ascending=False).head(20)


In [ ]:
# Visualisation des outliers sur la relation surface habitable vs prix
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Graphique 1 : GrLivArea vs SalePrice
axes[0].scatter(df['GrLivArea'], df['SalePrice'], alpha=0.4, c='steelblue', s=30)
# Mise en évidence des 2 anomalies principales
anomalies = df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 200000)]
axes[0].scatter(anomalies['GrLivArea'], anomalies['SalePrice'],
                color='red', s=80, zorder=5, label=f'{len(anomalies)} outliers détectés')
axes[0].set_title('Surface Habitable vs Prix de Vente', fontweight='bold')
axes[0].set_xlabel('GrLivArea (pieds carrés)')
axes[0].set_ylabel('SalePrice ($)')
axes[0].legend()

# Graphique 2 : Boxplot de LotArea (surface du terrain)
axes[1].boxplot(df['LotArea'].dropna(), vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Distribution de la Surface du Terrain (LotArea)', fontweight='bold')
axes[1].set_xlabel('LotArea (pieds carrés)')

plt.tight_layout()
plt.show()

print(f"Nombre d'outliers identifiés (grande surface + prix bas) : {len(anomalies)}")
print(anomalies[['GrLivArea', 'SalePrice', 'SaleCondition']])


In [ ]:
# Suppression des 2 outliers documentés
# Ces maisons (ventes atypiques : SaleCondition = Partial ou familiales)
# sont des cas particuliers qui nuiraient à l'entraînement du modèle.
df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 200000))]

print(f"Dataset après suppression des outliers : {df.shape[0]} lignes x {df.shape[1]} colonnes")


## 5. Distinction Variables Numériques et Catégorielles

Les colonnes se divisent en deux grandes familles :

- **Variables numériques continues** : Des mesures chiffrées réelles comme la
  surface en pieds carrés, le prix, l'année de construction, etc.
- **Variables numériques discrètes** : Des comptages entiers comme le nombre de
  chambres, de salles de bain, de cheminées, etc.
- **Variables catégorielles nominales** : Des catégories sans ordre (le quartier,
  le type de toit, le type de fondation). On ne peut pas dire que "Pave > Grvl".
- **Variables catégorielles ordinales** : Des catégories avec un ordre de qualité
  clair (Ex > Gd > TA > Fa > Po = Excellent > Bon > Moyen > Faible > Mauvais).


In [ ]:
# Séparation des colonnes par type
cols_numeriques = df.select_dtypes(include=[np.number]).columns.tolist()
cols_categorielles = df.select_dtypes(include=['object']).columns.tolist()

# On retire la variable cible de la liste des numériques
if 'SalePrice' in cols_numeriques:
    cols_numeriques.remove('SalePrice')

print(f"Variables NUMERIQUES ({len(cols_numeriques)}) :")
print(cols_numeriques)
print(f"\nVariables CATEGORIELLES ({len(cols_categorielles)}) :")
print(cols_categorielles)


In [ ]:
# Variables ordinales de qualité (ont un ordre de mérite clair)
COLS_ORDINALES = [
    'ExterQual', 'ExterCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2',
    'HeatingQC', 'KitchenQual',
    'FireplaceQu',
    'GarageQual', 'GarageCond', 'GarageFinish',
    'PoolQC', 'Functional',
    'PavedDrive', 'Fence'
]

# Variables nominales (sans ordre)
COLS_NOMINALES = [c for c in cols_categorielles if c not in COLS_ORDINALES]

print(f"Variables ordinales (avec ordre de qualité) : {len(COLS_ORDINALES)}")
print(COLS_ORDINALES)
print(f"\nVariables nominales (catégories sans ordre) : {len(COLS_NOMINALES)}")
print(COLS_NOMINALES)


## 6. Analyse Approfondie de `SalePrice` (Variable Cible)

`SalePrice` est la valeur que nos modèles devront prédire.
Il est fondamental de comprendre sa distribution avant de commencer.

- **Asymétrie à droite** : La majorité des maisons coûtent entre 130 000 $ et 220 000 $,
  mais quelques propriétés d'exception atteignent plus de 600 000 $.
  Cette queue à droite gêne les modèles de régression linéaire.

- **Solution : la transformation logarithmique `log(1 + x)`** :
  En travaillant sur le logarithme du prix, la distribution devient symétrique
  et les modèles font des prédictions nettement meilleures.
  Pour récupérer le prix réel depuis une prédiction, on appliquera `np.expm1()`.


In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Distribution brute
sns.histplot(df['SalePrice'], kde=True, color='royalblue', ax=axes[0], bins=45)
axes[0].set_title('Distribution Brute du Prix', fontweight='bold')
axes[0].set_xlabel('SalePrice ($)')
axes[0].axvline(df['SalePrice'].median(), color='red', linestyle='--', linewidth=2,
                label=f"Médiane : {df['SalePrice'].median():,.0f} $")
axes[0].legend(fontsize=9)

# 2. Distribution Log
log_price = np.log1p(df['SalePrice'])
sns.histplot(log_price, kde=True, color='darkorange', ax=axes[1], bins=45)
axes[1].set_title('Après Transformation log(1 + Prix)', fontweight='bold')
axes[1].set_xlabel('log(1 + SalePrice)')

# 3. QQ-Plot pour vérifier la normalité
(osm, osr), (slope, intercept, r) = stats.probplot(log_price, dist='norm', plot=None)
axes[2].plot(osm, osr, 'o', alpha=0.3, color='teal', markersize=4)
axes[2].plot(osm, slope*np.array(osm)+intercept, 'r-', linewidth=2)
axes[2].set_title('QQ-Plot (log du Prix vs Distribution Normale)', fontweight='bold')
axes[2].set_xlabel('Quantiles Théoriques')
axes[2].set_ylabel('Quantiles Observés')

plt.tight_layout()
plt.show()

print(f"Asymétrie (Skewness) brute        : {df['SalePrice'].skew():.3f}")
print(f"Asymétrie (Skewness) après log    : {log_price.skew():.3f}   (proche de 0 = bien symétrique)")
print(f"Corrélation avec la droite normale: {r:.4f}")


## 7. Séparation Train / Test — Avant Tout Prétraitement

### La règle d'or pour éviter la fuite de données (Data Leakage)

La fuite de données est l'une des erreurs les plus graves en Machine Learning.
Elle se produit lorsque des informations des données de test influencent
l'entraînement du modèle, ce qui donne une fausse impression de bonnes performances.

**Exemple concret de fuite :**
Si on calcule la médiane de `LotFrontage` sur TOUT le dataset pour l'imputation,
les données de test ont participé au calcul de cette médiane.
Le modèle aura "vu" indirectement les données de test pendant l'entraînement.

**La bonne pratique :**
1. Séparer en `X_train`, `X_test`, `y_train`, `y_test` en PREMIER.
2. Calculer toutes les statistiques (moyennes, médianes, fréquences) sur `X_train` uniquement.
3. Appliquer (transformer) ces statistiques sur `X_test` sans recalculer quoi que ce soit.

En pratique, le `Pipeline` scikit-learn gère tout cela automatiquement et correctement.


In [ ]:
# Séparation X / y
# On applique la transformation log sur la cible pour améliorer les prédictions
X = df.drop(columns=['SalePrice'])
y = np.log1p(df['SalePrice'])   # log(1 + prix) pour symétriser la distribution

print(f"X (caractéristiques) : {X.shape}")
print(f"y (cible log-transformée) : {y.shape}")
print(f"\nValeur min de y : {y.min():.3f}  |  Max : {y.max():.3f}")
print(f"Pour récupérer le prix réel depuis une prédiction yhat : np.expm1(yhat)")


In [ ]:
# Split 80% entraînement / 20% test avec graine aléatoire fixée
# random_state=42 garantit que le split est identique à chaque exécution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Données d'entraînement : {X_train.shape[0]} maisons ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Données de test        : {X_test.shape[0]} maisons ({X_test.shape[0]/len(X)*100:.1f}%)")
print("\n=> Tout le prétraitement suivant sera appris sur X_train uniquement.")


## 8. Traitement des Valeurs Manquantes

Nous appliquons une stratégie de remplacement adaptée à chaque groupe de colonnes.
La règle est simple : le sens de chaque champ dicte la logique de remplacement.


In [ ]:
# On travaille sur une copie pour laisser X_train/X_test inchangés
# Les remplacements manuels ci-dessous s'appliquent AVANT la construction du Pipeline

def imputer_valeurs_manquantes(data):
    df_ = data.copy()

    # ── A. Équipements optionnels catégoriels ─────────────────────────────────
    # NA dans la description = l'équipement n'existe pas ("None" = aucun garage, etc.)
    cols_none = [
        'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
        'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
        'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
        'MasVnrType'
    ]
    for col in cols_none:
        if col in df_.columns:
            df_[col] = df_[col].fillna('None')

    # ── B. Équipements optionnels numériques ──────────────────────────────────
    # Pas de garage => 0 voiture, 0 m², etc.
    cols_zero = [
        'GarageCars', 'GarageArea',
        'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
        'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
    ]
    for col in cols_zero:
        if col in df_.columns:
            df_[col] = df_[col].fillna(0.0)

    # ── C. GarageYrBlt : si pas de garage → année de construction de la maison
    if 'GarageYrBlt' in df_.columns and 'YearBuilt' in df_.columns:
        df_['GarageYrBlt'] = df_['GarageYrBlt'].fillna(df_['YearBuilt'])

    # ── D. LotFrontage : médiane par quartier (train uniquement pour éviter la fuite)
    # Note : dans le Pipeline final, on utilisera SimpleImputer(strategy='median')
    # Ici on fait une imputation rapide pour l'exploration
    if 'LotFrontage' in df_.columns and 'Neighborhood' in df_.columns:
        df_['LotFrontage'] = df_.groupby('Neighborhood')['LotFrontage'].transform(
            lambda x: x.fillna(x.median())
        )
        df_['LotFrontage'] = df_['LotFrontage'].fillna(df_['LotFrontage'].median())

    # ── E. Variables rares avec oubli ponctuel → mode (valeur la plus fréquente)
    cols_mode = [
        'MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd',
        'Electrical', 'KitchenQual', 'Functional', 'SaleType'
    ]
    for col in cols_mode:
        if col in df_.columns and df_[col].isnull().any():
            df_[col] = df_[col].fillna(df_[col].mode()[0])

    return df_

# Application sur les données d'entraînement
X_train_imp = imputer_valeurs_manquantes(X_train)
# Application sur les données de test (aucun recalcul, mêmes règles)
X_test_imp  = imputer_valeurs_manquantes(X_test)

# Vérification
null_train = X_train_imp.isnull().sum().sum()
null_test  = X_test_imp.isnull().sum().sum()
print(f"Valeurs manquantes restantes dans X_train : {null_train}")
print(f"Valeurs manquantes restantes dans X_test  : {null_test}")


## 9. Feature Engineering — Création de Nouvelles Colonnes

Le Feature Engineering consiste à créer de nouvelles variables synthétiques
à partir des colonnes existantes, pour donner au modèle des informations
plus directes et pertinentes.

Ces colonnes synthétiques représentent des notions que le marché immobilier
mesure naturellement : surface totale, âge réel, nombre de salles de bain total, etc.


In [ ]:
def feature_engineering(data):
    df_ = data.copy()

    # Surface totale bâtie (rez-de-chaussée + 1er étage + sous-sol)
    # C'est la variable la plus corrélée avec le prix après transformation
    df_['TotalSF'] = df_['GrLivArea'] + df_['TotalBsmtSF']

    # Nombre total de salles de bain en équivalent-plein (une demi-SdB compte 0.5)
    df_['TotalBaths'] = (
        df_['FullBath']
        + 0.5 * df_['HalfBath']
        + df_['BsmtFullBath']
        + 0.5 * df_['BsmtHalfBath']
    )

    # Âge réel de la maison au moment de la vente
    df_['HouseAge'] = df_['YrSold'] - df_['YearBuilt']
    df_['HouseAge'] = df_['HouseAge'].clip(lower=0)  # Sécurité : âge jamais négatif

    # La maison a-t-elle été rénovée ? (1 = oui, 0 = non)
    df_['IsRemodeled'] = (df_['YearRemodAdd'] > df_['YearBuilt']).astype(int)

    # Années depuis la dernière rénovation au moment de la vente
    df_['YearsSinceRemod'] = df_['YrSold'] - df_['YearRemodAdd']
    df_['YearsSinceRemod'] = df_['YearsSinceRemod'].clip(lower=0)

    # Surface des espaces extérieurs (terrasses, porches)
    df_['TotalPorchSF'] = (
        df_['WoodDeckSF'] + df_['OpenPorchSF']
        + df_['EnclosedPorch'] + df_['3SsnPorch'] + df_['ScreenPorch']
    )

    # La maison a-t-elle un sous-sol ? (1 = oui, 0 = non)
    df_['HasBasement'] = (df_['TotalBsmtSF'] > 0).astype(int)

    # La maison a-t-elle un garage ? (1 = oui, 0 = non)
    df_['HasGarage'] = (df_['GarageArea'] > 0).astype(int)

    # La maison a-t-elle une piscine ? (1 = oui, 0 = non)
    df_['HasPool'] = (df_['PoolArea'] > 0).astype(int)

    return df_

X_train_fe = feature_engineering(X_train_imp)
X_test_fe  = feature_engineering(X_test_imp)

nouvelles_cols = ['TotalSF', 'TotalBaths', 'HouseAge', 'IsRemodeled',
                  'YearsSinceRemod', 'TotalPorchSF', 'HasBasement', 'HasGarage', 'HasPool']
print(f"Colonnes créées par Feature Engineering ({len(nouvelles_cols)}) :")
for c in nouvelles_cols:
    print(f"  - {c}")

print(f"\nNouvelles dimensions de X_train : {X_train_fe.shape}")


In [ ]:
# Visualisation des nouvelles variables les plus importantes
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# TotalSF vs Prix (sur les données train uniquement)
train_plot = X_train_fe.copy()
train_plot['SalePrice'] = np.expm1(y_train)

axes[0].scatter(train_plot['TotalSF'], train_plot['SalePrice'],
                alpha=0.4, color='steelblue', s=20)
axes[0].set_title('Surface Totale (TotalSF) vs Prix', fontweight='bold')
axes[0].set_xlabel('TotalSF (pieds carrés)')
axes[0].set_ylabel('SalePrice ($)')

# HouseAge vs Prix
axes[1].scatter(train_plot['HouseAge'], train_plot['SalePrice'],
                alpha=0.4, color='darkorange', s=20)
axes[1].set_title('Âge de la Maison vs Prix', fontweight='bold')
axes[1].set_xlabel('HouseAge (années)')
axes[1].set_ylabel('SalePrice ($)')

# TotalBaths vs Prix (boxplot)
axes[2].boxplot(
    [train_plot[train_plot['TotalBaths'] == b]['SalePrice'].dropna()
     for b in sorted(train_plot['TotalBaths'].unique())],
    labels=[str(b) for b in sorted(train_plot['TotalBaths'].unique())],
    patch_artist=True,
    boxprops=dict(facecolor='lightgreen')
)
axes[2].set_title('Total Salles de Bain vs Prix', fontweight='bold')
axes[2].set_xlabel('TotalBaths')
axes[2].set_ylabel('SalePrice ($)')

plt.tight_layout()
plt.show()


## 10. Encodage des Variables Catégorielles

### Pourquoi encoder ?
Les algorithmes de Machine Learning calculent uniquement avec des nombres.
Il faut donc convertir tous les textes en valeurs numériques.

### Deux méthodes selon le type de variable :

**A. Variables Ordinales** (avec ordre de qualité) :
On leur attribue une note correspondant à leur rang (0 à 5, ou 0 à 7 selon la variable).
Exemple : `KitchenQual` : `Po → 1`, `Fa → 2`, `TA → 3`, `Gd → 4`, `Ex → 5`.

**B. Variables Nominales** (sans ordre hiérarchique) :
On utilise le **One-Hot Encoding** : chaque catégorie devient une colonne binaire (0 ou 1).
Exemple : `Neighborhood_OldTown = 1` si la maison est dans Old Town, sinon 0.


In [ ]:
# ── A. Encodage Ordinal (variables de qualité) ───────────────────────────────

# Dictionnaires de correspondance pour chaque variable ordinale
echelle_qualite     = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
echelle_exposition  = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
echelle_finition    = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
echelle_garage_fin  = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
echelle_pave_drive  = {'N': 0, 'P': 1, 'Y': 2}
echelle_fence       = {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}
echelle_functional  = {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4,
                       'Min2': 5, 'Min1': 6, 'Typ': 7}

mapping_ordinal = {
    'ExterQual': echelle_qualite,
    'ExterCond': echelle_qualite,
    'BsmtQual': echelle_qualite,
    'BsmtCond': echelle_qualite,
    'HeatingQC': echelle_qualite,
    'KitchenQual': echelle_qualite,
    'FireplaceQu': echelle_qualite,
    'GarageQual': echelle_qualite,
    'GarageCond': echelle_qualite,
    'PoolQC': echelle_qualite,
    'BsmtExposure': echelle_exposition,
    'BsmtFinType1': echelle_finition,
    'BsmtFinType2': echelle_finition,
    'GarageFinish': echelle_garage_fin,
    'PavedDrive': echelle_pave_drive,
    'Fence': echelle_fence,
    'Functional': echelle_functional,
}

def encoder_ordinaux(data):
    df_ = data.copy()
    for col, mapping in mapping_ordinal.items():
        if col in df_.columns:
            df_[col] = df_[col].map(mapping).fillna(0).astype(int)
    return df_

X_train_ord = encoder_ordinaux(X_train_fe)
X_test_ord  = encoder_ordinaux(X_test_fe)

print("Encodage ordinal appliqué. Exemple de résultat :")
X_train_ord[list(mapping_ordinal.keys())[:6]].head(3)


In [ ]:
# ── B. Encodage One-Hot (variables nominales) ────────────────────────────────
# On utilise pd.get_dummies pour créer les colonnes binaires
# On aligne train et test pour avoir exactement les mêmes colonnes après encodage

# Identification des colonnes nominales restantes après encodage ordinal
cols_nominales_restantes = [c for c in X_train_ord.select_dtypes(include='object').columns]
print(f"Colonnes nominales à encoder avec One-Hot : {len(cols_nominales_restantes)}")
print(cols_nominales_restantes)

# One-Hot Encoding
X_train_enc = pd.get_dummies(X_train_ord, columns=cols_nominales_restantes, drop_first=True)
X_test_enc  = pd.get_dummies(X_test_ord,  columns=cols_nominales_restantes, drop_first=True)

# Alignement des colonnes (le test peut manquer de catégories rares)
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

print(f"\nDimensions X_train après One-Hot : {X_train_enc.shape}")
print(f"Dimensions X_test  après One-Hot  : {X_test_enc.shape}")
print("\nLes deux datasets ont exactement les mêmes colonnes :", X_train_enc.shape[1] == X_test_enc.shape[1])


## 11. Normalisation et Standardisation

La normalisation est **indispensable pour les modèles linéaires** (Régression Linéaire, Ridge, Lasso)
car ils sont sensibles à l'échelle des variables.
- Une colonne `GrLivArea` entre 500 et 5 000 pieds carrés vs une colonne `FullBath` entre 0 et 4
  donneraient des coefficients incomparables sans normalisation.

Elle est **inutile pour les modèles à base d'arbres** (Random Forest, Gradient Boosting)
car ces modèles utilisent uniquement des seuils de coupure, pas des distances.

### Deux scalers disponibles :
- **`StandardScaler`** : Centre les données (moyenne = 0) et les réduit (écart-type = 1).
  Idéal pour les distributions proches de la normale.
- **`RobustScaler`** : Utilise la médiane et les quartiles. Robuste aux valeurs aberrantes.
  Préférable ici car certaines colonnes (LotArea, MiscVal) ont des queues très longues.

On utilise `RobustScaler` en priorité car notre dataset contient quelques outliers résiduels.


In [ ]:
# On récupère les noms de colonnes numériques après tout l'encodage
cols_num_finales = X_train_enc.select_dtypes(include=[np.number]).columns.tolist()

print(f"Nombre de colonnes numériques à normaliser : {len(cols_num_finales)}")

# Instanciation du scaler — entraîné sur X_train uniquement
scaler = RobustScaler()

# Fit sur X_train uniquement (calcule médiane et IQR du train)
X_train_scaled = X_train_enc.copy()
X_test_scaled  = X_test_enc.copy()

X_train_scaled[cols_num_finales] = scaler.fit_transform(X_train_enc[cols_num_finales])

# Transform sur X_test (utilise les statistiques calculées sur X_train)
X_test_scaled[cols_num_finales]  = scaler.transform(X_test_enc[cols_num_finales])

print(f"Entraînement : {X_train_scaled.shape}")
print(f"Test         : {X_test_scaled.shape}")
print("\nAperçu des données normalisées (train) :")
X_train_scaled[['GrLivArea', 'TotalSF', 'LotArea', 'HouseAge']].describe().round(3)


## 12. Synthèse Finale et Sauvegarde

Récapitulatif de toutes les transformations appliquées dans cette étape.


In [ ]:
print("=" * 60)
print("       RÉCAPITULATIF DE L'ÉTAPE 1 — PRÉPARATION DES DONNÉES")
print("=" * 60)
print(f"\nDataset initial (après suppression Id) :")
print(f"  - Lignes           : {len(df)}")
print(f"  - Colonnes         : {df.shape[1]} (dont 1 variable cible : SalePrice)")

print(f"\nAprès nettoyage et suppression des outliers :")
print(f"  - Outliers supprimés  : 2 (grande surface + prix aberrant)")

print(f"\nSplit Train / Test (80% / 20%, random_state=42) :")
print(f"  - X_train : {X_train_scaled.shape[0]} lignes x {X_train_scaled.shape[1]} colonnes")
print(f"  - X_test  : {X_test_scaled.shape[0]} lignes x {X_test_scaled.shape[1]} colonnes")
print(f"  - y_train : {y_train.shape[0]} valeurs log-transformées")
print(f"  - y_test  : {y_test.shape[0]} valeurs log-transformées")

print(f"\nTransformations appliquées :")
print(f"  - Valeurs manquantes imputées (None, 0, médiane quartier, mode)")
print(f"  - Feature Engineering : 9 nouvelles colonnes créées")
print(f"  - Encodage ordinal    : {len(mapping_ordinal)} colonnes de qualité → notes 0 à 7")
print(f"  - Encodage One-Hot    : {len(cols_nominales_restantes)} colonnes nominales")
print(f"  - Normalisation       : RobustScaler sur {len(cols_num_finales)} colonnes numériques")
print(f"  - Cible y             : np.log1p(SalePrice) pour symétriser la distribution")

print(f"\nAucune fuite de données (Data Leakage) :")
print(f"  - Le scaler et les statistiques d'imputation ont été calculés")
print(f"    exclusivement sur X_train.")
print(f"  - X_test a uniquement été transformé (pas appris).")
print("=" * 60)


In [ ]:
# Sauvegarde des datasets préparés pour les étapes suivantes (5, 6, 7, 8)
import joblib
import os

os.makedirs('data_prepared', exist_ok=True)

X_train_scaled.to_csv('data_prepared/X_train.csv', index=False)
X_test_scaled.to_csv( 'data_prepared/X_test.csv',  index=False)
y_train.to_csv('data_prepared/y_train.csv', index=False)
y_test.to_csv( 'data_prepared/y_test.csv',  index=False)

# Sauvegarde du scaler pour l'utiliser dans l'application Streamlit (Étape 8)
joblib.dump(scaler, 'data_prepared/robust_scaler.pkl')

print("Fichiers sauvegardés dans le dossier 'data_prepared/' :")
for f in os.listdir('data_prepared'):
    taille = os.path.getsize(f'data_prepared/{f}')
    print(f"  - {f:35s}  ({taille/1024:.1f} Ko)")


## Prochaines Étapes

Les données sont maintenant entièrement préparées.

- **Étape 5** : Entraînement des modèles (Régression Linéaire, Ridge, Random Forest, Gradient Boosting)
  avec validation croisée K-Fold et optimisation GridSearchCV.
- **Étape 6** : Évaluation et comparaison des modèles (MAE, RMSE, R²), graphiques Prix Réel vs Prédit.
- **Étape 7** : Interprétation des modèles (importances des variables, coefficients Ridge).
- **Étape 8** : Application Streamlit pour prédire le prix d'un logement en temps réel.
- **Étape 9** : Conteneurisation Docker pour déployer l'application partout.
